# COMP0178 - Project Tutorial 3: Time-saving Development Tips

- Source: [Project Tutorial 3](../project_tutorials/Project_Tutorial_3.pdf), 21 PDF pages, Dr Yuzuko Nakamura.
- Read one section at a time. Focus on making the database project reproducible and easier to debug.
- Slide explanations, supplementary practice, and version-specific corrections are labelled separately.
- Source checked against supplied `Lab3.pdf` on 6 November 2026: identical file. This confirms the supplied version, not the latest Moodle publication.
- [All project tutorials and source audit](project_tutorials.ipynb).

## 1. Debug the correct layer - PDF pages 2-5

- **View source** shows the HTML response received by the browser. It does not reveal the PHP source executed on the server.
- **Inspect / Elements** shows the current DOM and applied CSS. JavaScript may have changed this after the response arrived.
- **Console** shows browser-side errors and messages. Useful for JavaScript failures; a PHP or SQL failure may instead require server-side logs.
- Supplementary debugging example: if messages are stored but invisible, inspect the returned HTML/CSS. If nothing is stored, follow the request through PHP to its database operation.
- Temporary edits in developer tools help diagnose problems; save the actual fix in your project files.

## 2. Use Git to share reproducible changes - PDF pages 6-11

- A repository records snapshots of files and lets teammates combine changes.
- `git clone <repository-url>` makes a local copy. Use your actual project repository URL.
- `git add <filename>` stages selected changes; `git commit -m "Explain the change"` records them locally; `git push` publishes local commits.
- Correction to page 11: **add stages; commit records; push publishes**. Committing is not staging.
- `git pull` brings in remote changes and integrates them according to the configured strategy. If changes conflict, inspect both versions, resolve the intended result, then record the resolution.
- Supplementary project habit: commit schema changes together with the application changes that depend on them. Do not commit real passwords.
- Git tracks files, not automatically the live rows in MySQL. Keep SQL scripts and database backups as separate parts of reproducibility.

## 3. Keep database setup in SQL scripts - PDF pages 12-15

- MySQL console, phpMyAdmin, and Workbench are different interfaces to the database engine.
- Record table definitions in `schema.sql`; put repeatable sample records in `seed.sql`.
- A teammate should be able to create a fresh practice database from those files, rather than reconstructing your clicks in phpMyAdmin.
- `CREATE TABLE` defines columns and constraints. `PRIMARY KEY` uniquely identifies a row; `NOT NULL` requires a value.

```sql
-- MySQL teaching example: run inside your chosen practice database.
CREATE TABLE Users (
    id INTEGER AUTO_INCREMENT PRIMARY KEY,
    first_name VARCHAR(40) NOT NULL,
    family_name VARCHAR(40) NOT NULL
) ENGINE = InnoDB;
```

- `AUTO_INCREMENT` supplies identifiers when inserts omit `id`.
- Page 14 starts with `DROP DATABASE`: that deletes a database and its contents. It is not a necessary first step for normal development. The notebook practice below uses an isolated in-memory database instead.

## 4. Version-specific corrections to the setup slides - PDF pages 14-16

- The slide's combined `GRANT ... IDENTIFIED BY` is legacy syntax. In **MySQL 8.4**, create an account with `CREATE USER`, then assign its required privileges with `GRANT`. These are administrator setup operations, not commands the website needs on each request.
- For a new MySQL 8.4 schema, choose `utf8mb4` explicitly rather than copying the slide's legacy `utf8` spelling. Check your actual MySQL/MariaDB version before selecting version-specific options.
- **InnoDB** supports transactions, row-level locking, and foreign keys. **MyISAM** does not provide transactions or foreign-key enforcement.
- Page 16 says InnoDB has no full-text search. This is outdated: MySQL 8.4 supports full-text indexes for both InnoDB and MyISAM.
- Do not learn “InnoDB is slower” as a universal rule. Performance depends on workload and configuration; transaction and integrity requirements matter.

Version references: [MySQL 8.4 account management](https://dev.mysql.com/doc/refman/8.4/en/creating-accounts.html), [CREATE USER](https://dev.mysql.com/doc/refman/8.4/en/create-user.html), [full-text support](https://dev.mysql.com/doc/refman/8.4/en/fulltext-restrictions.html), [utf8mb4](https://dev.mysql.com/doc/refman/8.4/en/charset-unicode-utf8mb4.html).

## 5. Seed, load, and inspect the database - PDF pages 17-19

- Seed data gives the team a known starting point. Use ordinary straight SQL quote characters; the slide's curly closing quotes should not be copied.

```sql
INSERT INTO Users (first_name, family_name)
VALUES ('Ada', 'Lovelace'), ('Grace', 'Hopper');
SELECT * FROM Users ORDER BY id;
```

- In the **MySQL client**, `source schema.sql` executes a script. `source` is a client command, not portable SQL for every database interface.
- phpMyAdmin offers SQL and Import interfaces; select the intended database first.
- `SHOW DATABASES;` lists databases; `SHOW TABLES;` lists tables in the current database; `DESCRIBE Users;` shows column definitions.
- Supplementary checks: inspect constraints as well as column names, and confirm expected rows after loading. Re-running seed inserts can duplicate rows unless the script is deliberately designed otherwise.

## 6. Backups and the tutorial action list - PDF pages 20-21

- A SQL dump records schema and/or data, depending on export settings. phpMyAdmin Export produces a file; Import restores it.
- A backup is useful only if you can restore it. Supplementary practice: restore into a separate disposable database and verify tables, row counts, and relationships.
- A Git copy of `schema.sql` cannot recover live records that were never exported.
- Page 21 asks for a draft ER diagram, a shared repository, and experimentation with database scripts. This is the supplied slide's historical action list, not a newly verified deadline or instruction to upload anything.
- For current deliverables and dates, consult the applicable coursework brief/Moodle.

## 7. Runnable practice - supplementary SQLite example

- This uses Python's built-in SQLite in memory; it requires no MySQL server and changes no project database.
- SQLite uses `INTEGER PRIMARY KEY` here rather than MySQL's `AUTO_INCREMENT`, and has no `ENGINE = InnoDB`, `CREATE USER`, or `GRANT` syntax.
- Question: can a fresh database be built from scripts, populated, and checked automatically?

In [1]:
import sqlite3
schema = """
CREATE TABLE Users (
    id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    family_name TEXT NOT NULL
);
CREATE TABLE Messages (
    id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL REFERENCES Users(id),
    body TEXT NOT NULL
);
"""
seed = """
INSERT INTO Users VALUES (1, 'Ada', 'Lovelace'), (2, 'Grace', 'Hopper');
INSERT INTO Messages VALUES (1, 1, 'Hello from Ada');
"""
db = sqlite3.connect(':memory:')
db.execute('PRAGMA foreign_keys = ON')
db.executescript(schema)
db.executescript(seed)
rows = db.execute("""
    SELECT u.first_name, m.body
    FROM Users u JOIN Messages m ON m.user_id = u.id
    ORDER BY m.id
""").fetchall()
assert rows == [('Ada', 'Hello from Ada')]
print(rows)

[('Ada', 'Hello from Ada')]


### Practice: check that the relationship is enforced

- Predict: should a message belonging to nonexistent user 99 be accepted?
- Answer: no. A foreign key prevents a message from referring to a user absent from `Users`.
- The following cell deliberately attempts an invalid insert in the disposable database.

In [2]:
try:
    db.execute("INSERT INTO Messages VALUES (?, ?, ?)", (2, 99, 'Invalid user'))
except sqlite3.IntegrityError:
    print('Expected: foreign key rejected the nonexistent user.')
else:
    raise AssertionError('Foreign-key enforcement should reject user 99.')
assert db.execute('SELECT COUNT(*) FROM Messages').fetchone()[0] == 1

Expected: foreign key rejected the nonexistent user.


### Practice: restore a backup and check the result

- `iterdump()` is SQLite's demonstration mechanism here, not a MySQL command.
- The check compares actual restored results, rather than merely checking that a dump file exists.

In [3]:
dump = '\n'.join(db.iterdump())
restored = sqlite3.connect(':memory:')
restored.executescript(dump)
restored.execute('PRAGMA foreign_keys = ON')
assert restored.execute('SELECT * FROM Users ORDER BY id').fetchall() == db.execute('SELECT * FROM Users ORDER BY id').fetchall()
assert restored.execute('SELECT * FROM Messages ORDER BY id').fetchall() == db.execute('SELECT * FROM Messages ORDER BY id').fetchall()
assert restored.execute('PRAGMA foreign_key_check').fetchall() == []
print('Restored users, messages, and valid relationships.')
restored.close()
db.close()

Restored users, messages, and valid relationships.


## 8. Revision questions with answers

1. **Does View Source show PHP code?** No; it shows the returned page source.
2. **Does commit publish your changes?** No; push publishes commits to a remote.
3. **Why keep schema and seed scripts?** To rebuild a known database state and share it with teammates.
4. **Does Git automatically back up live database rows?** No.
5. **Why use foreign keys?** To enforce valid references between related tables.
6. **Why test a restore?** An export alone does not demonstrate that the needed data and structure can be recovered.
7. **Must InnoDB lack full-text search?** No; that statement in the old slide is outdated.

## 9. Exam connection and slide coverage

- [2024 Q1, PDF page 2](../Exampapers/COMP0178A7PF.pdf#page=2) asks for ER-to-relational-schema translation and a short explanation. Tutorial 3 helps turn a designed schema into reproducible DDL; the exam task itself is design, not a Git-command exercise.
- [2025 Q1, PDF page 2](../Exampapers/COMP0178A7PG.pdf#page=2) asks for a restaurant-review ER diagram with multiplicities. Define entities and relationships before implementing tables. Our Users/Messages practice is supplementary and is not that paper's solution.
- More exam preparation: [past-paper guide](../exam_guide.ipynb).

| PDF pages | Treatment |
|---|---|
| 1, 2, 6, 12 | Title and section dividers; no separate memorization needed |
| 3-5 | Browser debugging |
| 7-11 | Git workflow, with staging correction |
| 13-16 | Scripts, DDL, engines, and version corrections |
| 17-19 | Seed data, loading, and inspection |
| 20-21 | Backup/restore and historical project action list |